# Project 1500 — Runner

**Workflow:**
1. Set player config in cell 1 (`source`: `'chesscom'` | `'lichess'` | `'both'`)
2. Run cell 2 to download any new games (skips already-cached months)
3. Run cells 3-4 to see phase analysis + opening priority list
4. Edit `LESSON_SELECTIONS` in cell 5 with openings from the list
5. Run cell 6 to generate lesson PDFs

In [ ]:
import importlib, os
import lesson_runner, game_fetcher
importlib.reload(lesson_runner)
importlib.reload(game_fetcher)
from lesson_runner import run_phase_analysis, compute_opening_priority, run_lesson
from game_fetcher  import fetch_games, games_dir_for

# ── configure player ──────────────────────────────────────────────────────────
# games_dir  auto-derived: games_{username}_{source}
# output_dir auto-derived: output/{username}_{source}_{method}{depth}
#
# Date filtering (all optional, format: 'YYYY_MM'):
#   start_date    — ignore games before this month (inclusive)
#   end_date      — ignore games after this month (inclusive)
#   exclude_months — set of specific months to skip (for non-contiguous gaps)

# PLAYER = {
#     'username':       'jf4bes',
#     'source':         'chesscom',
#     'time_classes':   {'rapid', 'blitz'},
#     'start_date':     '2022_12',
#     'exclude_months': {'2025_01', '2025_02', '2025_03'},
# }

# ── other player presets (uncomment to switch) ────────────────────────────────

# TheNomNomFactor Chess.com bullet+blitz+rapid since 2015:
# PLAYER = {
#     'username':    'TheNomNomFactor',
#     'source':      'chesscom',
#     'time_classes': {'bullet', 'blitz', 'rapid'},
#     'start_date':  '2015_01',
# }

# quinnleventhal505 Chess.com rapid+daily+blitz:
# PLAYER = {
#     'username':       'quinnleventhal505',
#     'source':         'chesscom',
#     'time_classes':   {'rapid', 'daily', 'blitz'},
#     'exclude_months': set(),
# }

PLAYER = {
    'username':       'omansour6',
    'source':         'chesscom',
    'time_classes':   {'rapid', 'daily', 'blitz','bullet'},
    'start_date':       '2020_01',
}

# sfink37 Chess.com bullet+blitz:
# PLAYER = {
#     'username':       'sfink37',
#     'source':         'chesscom',
#     'time_classes':   {'bullet', 'blitz'},
# }

# WIP — Lichess (jforbes94): confirm username before running
# PLAYER = {
#     'username':         'jf4bes',
#     'lichess_username': 'jforbes94',   # ← verify at lichess.org/@/jforbes94
#     'source':           'lichess',
#     'time_classes':     {'rapid'},
#     'start_date':       '2020_01',
# }

# ── aggregation settings ──────────────────────────────────────────────────────
GROUP_BY       = 'eco'   # 'url' | 'eco'
PRIORITY_DEPTH = 4       # url mode: 3 = broad, 4 = specific
ECO_DEPTH      = 3       # eco mode: 2 = broad, 3 = standard

# Resolve directories — output folder encodes player + source + method + depth
PLAYER['games_dir'] = games_dir_for(PLAYER)
_depth_tag = f'eco{ECO_DEPTH}' if GROUP_BY == 'eco' else f'url{PRIORITY_DEPTH}'
OUTPUT_DIR = os.path.join('output', f'{PLAYER["username"]}_{PLAYER["source"]}_{_depth_tag}')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"games_dir:  {PLAYER['games_dir']}")
print(f"output_dir: {OUTPUT_DIR}")
# ─────────────────────────────────────────────────────────────────────────────

games_dir:  games_omarmansour6_chesscom
output_dir: output\omarmansour6_chesscom_eco3


In [152]:
# Step 0 — Download games (run once, or re-run to pick up new months)
# Downloads to PLAYER['games_dir'] — printed above in cell 1.
# Skips months already saved locally.
fetch_games(PLAYER)

Chess.com done: 0 new month(s), 1 already cached


In [153]:
# Step 1 — Phase analysis: where are games being decided?
phase = run_phase_analysis(PLAYER, output_dir=OUTPUT_DIR)

losses      = phase['losses']
total_loss  = len(losses)
from collections import Counter
phase_counts = Counter(g['phase'] for g in losses)

PHASES = ['Opening (<=move 10)', 'Middlegame (move 11-25)', 'Endgame (move 26+)']
print(f'\nPhase breakdown ({total_loss} losses):')
for ph in PHASES:
    n   = phase_counts[ph]
    pct = 100 * n / total_loss
    bar = '#' * int(pct / 2)
    print(f'  {ph:30s} {n:4d}  ({pct:4.1f}%)  {bar}')

dominant = max(PHASES, key=lambda p: phase_counts[p])
print(f'\nDominant loss phase: {dominant}')

timeout_pct = 100 * sum(1 for g in losses
                        if g['end_reason'] in ('timeout','abandoned')) / total_loss
if timeout_pct > 15:
    print(f'>> Time management: {timeout_pct:.0f}% of losses are timeouts')

opening_pct = 100 * phase_counts[PHASES[0]] / total_loss
if opening_pct > 25:
    print('>> Opening losses are significant — lesson generation is high value')
else:
    print('>> Opening losses are low — middlegame/endgame training may have more impact')

Phase analysis: loading all games for omarmansour6 ...
  2 games  100.0% win  0 losses
Unexpected exception formatting exception. Falling back to standard exception


Traceback (most recent call last):
  File "c:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 3526, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\jeffr\AppData\Local\Temp\ipykernel_20748\3159548959.py", line 2, in <module>
    phase = run_phase_analysis(PLAYER, output_dir=OUTPUT_DIR)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Github\chessanalysis\lesson_runner.py", line 842, in run_phase_analysis
    _generate_phase_pdf(games, config, pdf_path)
  File "e:\Github\chessanalysis\lesson_runner.py", line 725, in _generate_phase_pdf
    '  |  '.join(f'{p}: {100*phase_counts[p]/total_losses:.0f}%' for p in _PHASES),
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "e:\Github\chessanalysis\lesson_runner.py", line 725, in <genexpr>
    '  |  '.join(f'{p}: {100*phase_counts[p]/total_losses:.0f}%' for p in _PHASES),
                         ~~~~~~~~~~~~~~~~~~

<Figure size 1100x850 with 0 Axes>

In [ ]:
# Step 2 — Opening priority list (both colors, Wilson-weighted, sorted by impact)
import csv

priority_white = compute_opening_priority(PLAYER, min_games=10, depth=PRIORITY_DEPTH,
                                          color='white', group_by=GROUP_BY, eco_depth=ECO_DEPTH)
priority_black = compute_opening_priority(PLAYER, min_games=10, depth=PRIORITY_DEPTH,
                                          color='black', group_by=GROUP_BY, eco_depth=ECO_DEPTH)

for color, pri in [('WHITE', priority_white), ('BLACK', priority_black)]:
    mode = f'eco_depth={ECO_DEPTH}' if GROUP_BY == 'eco' else f'url_depth={PRIORITY_DEPTH}'
    print(f'\nAs {color} — top priority openings ({GROUP_BY}, {mode}):')
    if GROUP_BY == 'eco':
        print(f'  {"#":<4} {"ECO":<6} {"Opening":<40} {"G":>5} {"Win%":>6} {"Priority":>9}')
        print('  ' + '-' * 75)
        for i, op in enumerate(pri[:20], 1):
            print(f'  {i:<4} {op["eco_code"]:<6} {op["name"]:<40} {op["games"]:>5} {op["wr"]:>5.0f}%  {op["priority"]:>9.0f}')
    else:
        print(f'  {"#":<4} {"Opening":<45} {"G":>5} {"Win%":>6} {"Priority":>9}  ECO key')
        print('  ' + '-' * 85)
        for i, op in enumerate(pri[:20], 1):
            print(f'  {i:<4} {op["name"]:<45} {op["games"]:>5} {op["wr"]:>5.0f}%  {op["priority"]:>9.0f}  {op["eco_key"]}')

# ── save full priority lists to CSV ───────────────────────────────────────────
depth_tag = f'eco{ECO_DEPTH}' if GROUP_BY == 'eco' else f'url{PRIORITY_DEPTH}'
csv_path  = os.path.join(OUTPUT_DIR, f'priority_{PLAYER["username"]}_{GROUP_BY}_{depth_tag}.csv')

all_rows = []
for op in priority_white + priority_black:
    c = op['counts']
    all_rows.append({
        'color':    op['color'],
        'rank':     '',
        'name':     op['name'],
        'eco_code': op.get('eco_code') or '',
        'eco_key':  op.get('eco_key')  or '',
        'games':    op['games'],
        'wins':     c.get('win',  0),
        'losses':   c.get('loss', 0),
        'draws':    c.get('draw', 0),
        'win_pct':  round(op['wr'], 1),
        'priority': round(op['priority'], 1),
        'group_by': GROUP_BY,
        'depth':    ECO_DEPTH if GROUP_BY == 'eco' else PRIORITY_DEPTH,
    })

for color_label in ('white', 'black'):
    rank = 1
    for row in all_rows:
        if row['color'] == color_label:
            row['rank'] = rank
            rank += 1

fieldnames = ['color','rank','name','eco_code','eco_key','games','wins','losses','draws',
              'win_pct','priority','group_by','depth']
with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_rows)

print(f'\nSaved: {csv_path}  ({len(all_rows)} rows)')

# Combined priority list (for GENERATE_ALL)
priority = [dict(op, color='white') for op in priority_white] + \
           [dict(op, color='black') for op in priority_black]
priority.sort(key=lambda x: -x['priority'])

In [ ]:
# Step 3 — Select lessons to generate
# Set GENERATE_ALL = True to run the top TOP_N openings from the priority list automatically.
# Set GENERATE_ALL = False and edit LESSON_SELECTIONS to pick specific openings by name.

GENERATE_ALL = True
TOP_N        = 25    # only used when GENERATE_ALL = True

LESSON_SELECTIONS = [
    # 'Giuoco Piano Game Main',
    # 'Philidor Defense',
]

In [ ]:
# Step 4 — Generate lesson PDFs  →  output/{username}_{source}/lessons/
priority_index = {(op['name'], op['color']): op for op in priority}

if GENERATE_ALL:
    selections = [(op['name'], op['color']) for op in priority[:TOP_N]]
    print(f'GENERATE_ALL = True — running top {len(selections)} openings by priority')
else:
    selections = [(name, 'white') for name in LESSON_SELECTIONS]
    print(f'Running {len(selections)} manually selected opening(s)')

lesson_results = []
for i, (name, color) in enumerate(selections, 1):
    match = priority_index.get((name, color))
    if not match:
        print(f'  WARNING: "{name}" ({color}) not found in priority list')
        lesson_results.append({'name': name, 'pdf_path': None, 'consistent': []})
        continue
    print(f'\n{"-"*60}')
    lesson_config = {
        **PLAYER,
        'eco_key':         match['lesson_eco_key'],
        'eco_code_prefix': match['eco_code_prefix'],
        'lesson_title':    f'{name} (as {color})',
        'lesson_number':   i,
        'color':           color,
    }
    result = run_lesson(lesson_config, output_dir=OUTPUT_DIR)
    result['name']  = name
    result['color'] = color
    lesson_results.append(result)

In [ ]:
# Step 4b — Save run summary CSV  →  output/{username}_{source}/run_summary_*.csv
# Captures: priority rank, games loaded, raw/kept/skipped junctions, skip reasons.
# ⚠ = too diverse: ≥15 games but 0 raw junctions — real weakness, no lesson possible
#     at the current ECO grouping level.

import csv

depth_tag    = f'eco{ECO_DEPTH}' if GROUP_BY == 'eco' else f'url{PRIORITY_DEPTH}'
summary_path = os.path.join(OUTPUT_DIR,
    f'run_summary_{PLAYER["username"]}_{GROUP_BY}_{depth_tag}.csv')

priority_lookup = {(op['name'], op['color']): (i+1, op) for i, op in enumerate(priority)}

rows = []
for r in lesson_results:
    name  = r['name']
    color = r.get('color', 'white')
    rank, op = priority_lookup.get((name, color), (None, {}))

    skipped      = r.get('skipped', [])
    skip_reasons = ' | '.join(reason for _, _, _, reason in skipped) if skipped else ''
    too_diverse  = r.get('total', 0) >= 15 and len(r.get('junctions', [])) == 0

    rows.append({
        'priority_rank':        rank,
        'name':                 name,
        'color':                color,
        'eco_code':             op.get('eco_code') or op.get('eco_key') or '',
        'priority_score':       round(op.get('priority', 0), 1),
        'win_pct':              round(op.get('wr', 0), 1),
        'games_in_priority':    op.get('games', 0),
        'games_loaded':         r.get('total', 0),
        'raw_junctions':        len(r.get('junctions', [])),
        'consistent_junctions': len(r.get('consistent', [])),
        'skipped_junctions':    len(skipped),
        'skip_reasons':         skip_reasons,
        'too_diverse':          'yes' if too_diverse else '',
        'pdf_generated':        'yes' if r.get('pdf_path') else 'no',
    })

fieldnames = ['priority_rank','name','color','eco_code','priority_score','win_pct',
              'games_in_priority','games_loaded','raw_junctions','consistent_junctions',
              'skipped_junctions','skip_reasons','too_diverse','pdf_generated']

with open(summary_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f'Run summary: {summary_path}')
print()
print(f'  {"#":<4} {"Opening":<42} {"Cl":<6} {"G":>5} {"Raw":>4} {"OK":>4} {"Sk":>4}  {"PDF":<4}  Notes')
print('  ' + '-' * 105)
for row in rows:
    flag  = ' ⚠ too diverse' if row['too_diverse'] else ''
    notes = (row['skip_reasons'][:50] + flag) if row['skip_reasons'] else flag
    print(f'  {str(row["priority_rank"]):<4} {row["name"]:<42} {row["color"]:<6}'
          f' {row["games_loaded"]:>5} {row["raw_junctions"]:>4} {row["consistent_junctions"]:>4}'
          f' {row["skipped_junctions"]:>4}  {row["pdf_generated"]:<4}  {notes}')

In [ ]:
# Step 5 — Merge all PDFs into one report with TOC and White/Black sections
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as mpdf
from pypdf import PdfWriter, PdfReader
from datetime import datetime

# ── separate lessons by color, numbered sequentially ──────────────────────────
white_lessons = [(r['name'], r['pdf_path'])
                 for r in lesson_results
                 if r.get('pdf_path') and r.get('consistent') and r.get('color') == 'white']
black_lessons = [(r['name'], r['pdf_path'])
                 for r in lesson_results
                 if r.get('pdf_path') and r.get('consistent') and r.get('color') == 'black']

# ── build ordered section list ────────────────────────────────────────────────
sections = []   # (group, display_title, pdf_path)
if phase.get('pdf_path'):
    sections.append(('phase', 'Phase Analysis', phase['pdf_path']))
for i, (name, path) in enumerate(white_lessons, 1):
    sections.append(('white', f'W{i}.  {name}', path))
for i, (name, path) in enumerate(black_lessons, 1):
    sections.append(('black', f'B{i}.  {name}', path))

skipped = [r['name'] for r in lesson_results if not r.get('consistent')]
for s in skipped:
    print(f'  Skipped (no junctions): {s}')

if not sections:
    print('Nothing to merge.')
else:
    # ── pre-scan page counts to compute TOC page numbers ──────────────────────
    # TOC itself is page 1; everything else starts at page 2
    page_counts = {path: len(PdfReader(path).pages) for _, _, path in sections}
    toc_entries = []
    current = 2
    for group, title, path in sections:
        toc_entries.append((group, title, current))
        current += page_counts[path]
    total_pages = current - 1

    # ── generate TOC page ─────────────────────────────────────────────────────
    toc_pdf_path = os.path.join(OUTPUT_DIR, '_toc.pdf')
    fig = plt.figure(figsize=(11, 8.5))
    depth_tag = f'eco{ECO_DEPTH}' if GROUP_BY == 'eco' else f'url{PRIORITY_DEPTH}'
    fig.text(0.5, 0.93, f'Project 1500 — {PLAYER["username"]}',
             ha='center', fontsize=18, fontweight='bold')
    fig.text(0.5, 0.87, f'{depth_tag.upper()}  |  {", ".join(sorted(PLAYER["time_classes"]))}  |  {datetime.today().strftime("%Y-%m-%d")}',
             ha='center', fontsize=11, color='#666')

    ax = fig.add_axes([0.08, 0.08, 0.84, 0.72])
    ax.axis('off')

    y = 1.0
    group_colors = {'phase': '#333', 'white': '#1a6b3c', 'black': '#8b1a1a'}
    last_group = None

    for group, title, page in toc_entries:
        # Section header
        if group != last_group:
            if last_group is not None:
                y -= 0.04
            header = {'phase': 'ANALYSIS', 'white': 'WHITE OPENINGS', 'black': 'BLACK OPENINGS'}[group]
            ax.text(0, y, header, fontsize=10, fontweight='bold',
                    color=group_colors[group], transform=ax.transAxes)
            y -= 0.055
            last_group = group

        # Dot leader
        dots = '.' * max(5, 90 - len(title))
        ax.text(0.03, y, title, fontsize=9.5, fontfamily='monospace',
                color='#222', transform=ax.transAxes)
        ax.text(0.97, y, str(page), fontsize=9.5, fontfamily='monospace',
                ha='right', color='#444', transform=ax.transAxes)
        ax.text(0.03 + len(title) * 0.0068, y, dots, fontsize=9.5,
                fontfamily='monospace', color='#bbb', transform=ax.transAxes)
        y -= 0.052

    fig.text(0.5, 0.04, f'{total_pages} pages', ha='center', fontsize=9, color='#aaa')
    plt.savefig(toc_pdf_path, bbox_inches='tight')
    plt.close(fig)

    # ── assemble final PDF ────────────────────────────────────────────────────
    writer = PdfWriter()

    # TOC page
    for page in PdfReader(toc_pdf_path).pages:
        writer.add_page(page)
    writer.add_outline_item('Table of Contents', 0)

    # Sections with nested outline (White / Black parent items)
    outline_parents = {}
    for group, title, path in sections:
        chapter_page = len(writer.pages)
        for page in PdfReader(path).pages:
            writer.add_page(page)

        if group == 'phase':
            writer.add_outline_item(title, chapter_page)
        else:
            if group not in outline_parents:
                label = 'White Openings' if group == 'white' else 'Black Openings'
                outline_parents[group] = writer.add_outline_item(label, chapter_page)
            writer.add_outline_item(title, chapter_page, parent=outline_parents[group])

    merged_path = os.path.join(OUTPUT_DIR, f'report_{PLAYER["username"]}.pdf')
    with open(merged_path, 'wb') as f:
        writer.write(f)

    # Clean up temp TOC file
    os.remove(toc_pdf_path)

    print(f'\nReport: {merged_path}')
    print(f'  {len(white_lessons)} white lesson(s), {len(black_lessons)} black lesson(s)')
    print(f'  {total_pages} pages total')